In [0]:
%pip install pypdf
%pip install transformers
dbutils.library.restartPython()

In [0]:
import os
import pypdf
from typing import List, Dict
from transformers import AutoTokenizer
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import serving

In [0]:
TABLE_NAME = "workspace.default.pioneer_cdj_pdf_chunks"
pdf_directory_path = "/Volumes/workspace/default/pdf_docs/"

# The tokenizer model name registered in your Databricks workspace.
TOKENIZER_MODEL = "llama_v3_3_70b_instruct"

# Max tokens per chunk is now a hard limit.
MAX_TOKENS_PER_CHUNK = 500

# The separators for splitting text, in order of preference.
SEPARATORS = ["\n\n", "\n", " ", ""]

def get_pdf_text(pdf_path: str) -> str:
    """Reads all text from a PDF file."""
    text = ""
    try:
        with open(pdf_path, "rb") as file:
            reader = pypdf.PdfReader(file)
            for page in reader.pages:
                text += page.extract_text()
    except Exception as e:
        print(f"Error reading PDF file {pdf_path}: {e}")
    return text


def chunk_text(text: str, tokenizer: AutoTokenizer) -> List[str]:
    """
    Splits text into chunks under the token limit by progressively
    splitting with different separators. Ensures no chunk exceeds MAX_TOKENS_PER_CHUNK.
    """
    chunks = [text]
    for separator in SEPARATORS:
        new_chunks = []
        for chunk in chunks:
            # Check if the chunk is too large and needs to be split further
            if len(tokenizer.encode(chunk)) > MAX_TOKENS_PER_CHUNK:
                # Split the chunk by the current separator
                parts = chunk.split(separator)
                current_chunk = ""
                for part in parts:
                    combined = current_chunk + separator + part if current_chunk else part
                    if len(tokenizer.encode(combined)) <= MAX_TOKENS_PER_CHUNK:
                        current_chunk = combined
                    else:
                        if current_chunk:
                            new_chunks.append(current_chunk.strip())
                        # If part itself is too large, split it into smaller pieces
                        while len(tokenizer.encode(part)) > MAX_TOKENS_PER_CHUNK:
                            tokens = tokenizer.encode(part)
                            sub_part = tokenizer.decode(tokens[:MAX_TOKENS_PER_CHUNK])
                            new_chunks.append(sub_part.strip())
                            part = tokenizer.decode(tokens[MAX_TOKENS_PER_CHUNK:])
                            tokens = tokenizer.encode(part)
                        current_chunk = part.strip()
                if current_chunk:
                    new_chunks.append(current_chunk.strip())
            else:
                new_chunks.append(chunk.strip())
        chunks = new_chunks
    
    # Final pass: ensure no chunk exceeds MAX_TOKENS_PER_CHUNK
    final_chunks = []
    for chunk in chunks:
        tokens = tokenizer.encode(chunk)
        start = 0
        while start < len(tokens):
            sub_tokens = tokens[start:start+MAX_TOKENS_PER_CHUNK]
            sub_chunk = tokenizer.decode(sub_tokens)
            final_chunks.append(sub_chunk.strip())
            start += MAX_TOKENS_PER_CHUNK

    # Filter out any empty strings
    return [chunk for chunk in final_chunks if chunk]


def process_and_insert_chunks(spark, pdf_path: str, tokenizer: AutoTokenizer):
    """
    Processes a single PDF, chunks its text, and inserts it into the table.
    """
    print(f"--- Processing PDF: {pdf_path} ---")
    pdf_text = get_pdf_text(pdf_path)

    if not pdf_text:
        print("PDF text is empty, nothing to chunk.")
        return

    chunks = chunk_text(pdf_text, tokenizer)
    print(f"Found {len(chunks)} chunks.")

    try:
        # Create a list of tuples to represent the data
        data_to_insert = [(chunk,) for chunk in chunks]
        
        # Create a temporary DataFrame and write it to the table
        spark.createDataFrame(data_to_insert, ['chunk_text']).write.format("delta").mode("append").saveAsTable(TABLE_NAME)
        
        print(f"Insertion complete for {pdf_path}.")
    except Exception as e:
        print(f"Error interacting with Databricks: {e}")



# --- Main Execution Block ---

if not os.path.isdir(pdf_directory_path):
    print(f"Error: Directory not found at {pdf_directory_path}")
else:
    tokenizer = AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer", cache_dir="/tmp/hf_cache")

    print(f"Checking for table '{TABLE_NAME}'...")
    try:
        spark.sql(f"DESCRIBE TABLE {TABLE_NAME}")
        print(f"Table '{TABLE_NAME}' already exists. Deleting previous chunks...")
        spark.sql(f"TRUNCATE TABLE {TABLE_NAME}")
        # Enable Change Data Feed on the existing table
        spark.sql(f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    except Exception:
        print(f"Table '{TABLE_NAME}' not found. Creating a new table.")
        create_table_sql = f"""
        CREATE TABLE {TABLE_NAME} (
            id BIGINT GENERATED BY DEFAULT AS IDENTITY,
            chunk_text STRING
        )
        TBLPROPERTIES (delta.enableChangeDataFeed = true)
        """
        spark.sql(create_table_sql)
    print(f"Table '{TABLE_NAME}' ready for new data.")

    print(f"Processing all PDFs in directory: {pdf_directory_path}")
    
    for filename in os.listdir(pdf_directory_path):
        if filename.endswith(".pdf"):
            full_pdf_path = os.path.join(pdf_directory_path, filename)
            process_and_insert_chunks(spark, full_pdf_path, tokenizer)
    
    print("All PDFs in the directory have been processed.")